In [1]:
from pdf_analysis import PdfDetector
import os
import pandas as pd

pdf_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\RANDOM_FETCH_DATA\NSE_FETCH\ANNUAL_REPORTS_2025"

In [ ]:
from pathlib import Path
from PyPDF2 import PdfReader

# Folder containing PDFs
ROOT_FOLDER = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\RANDOM_FETCH_DATA\NSE_FETCH\ANNUAL_REPORTS_2025"
DELETE_CORRUPTED = True

root = Path(ROOT_FOLDER)
report_file = root / "Corrupted_PDFs.txt"

corrupted_files = []

print("Scanning PDFs...")

for pdf_file in root.rglob("*.pdf"):
    try:
        with open(pdf_file, "rb") as f:
            reader = PdfReader(f)

            # Force reading all pages
            _ = len(reader.pages)

    except Exception as e:
        corrupted_files.append((str(pdf_file), str(e)))
        print(f"CORRUPTED: {pdf_file}")

# Write report
with open(report_file, "w", encoding="utf-8") as f:
    f.write("CORRUPTED PDF FILES\n")
    f.write("=" * 100 + "\n\n")

    for file_path, error in corrupted_files:
        f.write(f"{file_path}\n")
        f.write(f"Error: {error}\n\n")

print(f"\nFound {len(corrupted_files)} corrupted PDFs.")
print(f"Report saved to: {report_file}")

if DELETE_CORRUPTED:
    print("\nDeleting corrupted PDFs...")
    for file_path, _ in corrupted_files:
        try:
            Path(file_path).unlink()
            print(f"Deleted: {file_path}")
        except Exception as e:
            print(f"Failed to delete {file_path}: {e}")

    print("Deletion completed.")

In [ ]:
from pdf_analysis import PdfDetector
import os
import pandas as pd

pdf_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\RANDOM_FETCH_DATA\NSE_FETCH\ANNUAL_REPORTS_2025"
detector = PdfDetector()
all_content = []
for file_name in os.listdir(pdf_path)[100:140]:
    print(file_name)
    file_path = os.path.join(pdf_path, file_name)
    
    result = detector.detect(file_path)
    temp = result.summary_dict()
    print(temp.keys())
    all_content.append(temp)
    # print(result.summary())
pd.DataFrame(all_content).to_csv("D2SAMPLE.csv")

In [ ]:
pd.DataFrame(all_content).to_csv("D2SAMPLE.csv")

In [ ]:
import pandas as pd
import os, json
import fitz
from pathlib import Path

path =r"PDF_PAGE.xlsx"
folder_pdf = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\RANDOM_FETCH_DATA\NSE_FETCH\ANNUAL_REPORTS_2026"
def page_text_or_scanned(page):

    text = page.get_text("text").strip()
    page_rect = page.rect
    page_area = page_rect.width * page_rect.height

    if page_area <= 0:
        return "scanned"

    image_area = 0
    for img in page.get_images(full=True):
        try:
            xref = img[0]
            for rect in page.get_image_rects(xref):
                clipped = rect & page_rect
                if clipped.is_empty:
                    continue

                rect_area = clipped.width * clipped.height
                # Ignore small logos/icons
                if rect_area > page_area * 0.05:
                    image_area += rect_area

        except Exception:
            continue

    image_coverage = min(image_area / page_area, 1.0)
    blocks = page.get_text("blocks")
    text_blocks = [
        block for block in blocks if len(block) >= 5 and str(block[4]).strip()
    ]

    num_text_blocks = len(text_blocks)

    # Strong text page
    if len(text) > 100 and num_text_blocks >= 3 and image_coverage < 0.8:
        return "text"
    # Strong scanned page
    if image_coverage > 0.8 and len(text) < 100:
        return "scanned"
    # OCR scanned page
    if image_coverage > 0.9 and num_text_blocks <= 2:
        return "scanned"
    return "text" if len(text) > 100 else "scanned"

df = pd.read_excel(path, sheet_name="2026")
df.head(4)

for file in os.listdir(folder_pdf):

    file_path = os.path.join(folder_pdf, file)

    stem = Path(file_path).name
    print(f"\nProcessing: {stem}")

    doc = fitz.open(file_path)

    mask = df["pdf_name"].str.contains(stem, na=False)

    # print("Matched rows:", mask.sum())

    for idx, row in df.loc[mask].iterrows():

        # print("Row:", idx)

        page_n = int(row["page_number"]) - 1

        res = page_text_or_scanned(doc[page_n])

        print("Result:", res)

        df.at[idx, "page_type"] = res

    doc.close()

print(df.head())

# Overwrite original file
df.to_excel(path, sheet_name="2026",index=False)

print(f"Saved updates to: {path}")

In [2]:
# Overwrite original file
df.to_excel(path, sheet_name="2026",index=False)

print(f"Saved updates to: {path}")

Saved updates to: PDF_PAGE.xlsx
